# Financial Analyst From Scratch — Claude-Code Harness + Live Research

A financial-analysis agent built on the same single-notebook Claude-Code harness
as `claude_code_from_scratch.ipynb`, extended with **live research tools** so it
can answer complex, time-sensitive financial questions instead of relying on
stale training data.

### What was added on top of the base harness

| Tool | Purpose |
|---|---|
| `tavily_search` | Web/news/macro research — filings, analyst views, economic data, qualitative context. Returns summarized sources **with URLs**. |
| `finnhub` | Live **structured** market data — quotes, fundamentals (P/E, margins, market cap, ROE), company profile, recent news. Exact numbers, no scraping. |
| `calc` | Financial math sandbox — all `math.*` plus `npv`, `irr`, `fv`, `pv`, `pmt`. |

Everything else is inherited from the base harness: the perception-action
`agent_loop`, `spawn_subagent` (isolated, gets the research tools too), `todo_*`
planning, the persistent `task_*` graph, rolling context compaction, skills, and
the `chat()` entry point.

### Design intent

Complex financial questions need **decomposition + current data + arithmetic +
synthesis**. The lead model (`qwen3:32b`) plans with `todo_write`, pulls exact
figures from `finnhub`, gathers context/news with `tavily_search`, runs the
numbers with `calc`, delegates noisy multi-source gathering to subagents, and
ends with a sourced answer.

> Both `tavily_search` and `finnhub` are called over raw HTTP via `requests`
> (no extra packages). Keys are read from the project `.env`
> (`TAVILY_API_KEY`, `FINNHUB_API_KEY`).

> **Not financial advice.** Output is informational and may contain errors —
> always verify figures against primary sources.


In [ ]:
"""
Imports + logging.

All logging goes through a single logger named "agent". Every subsystem uses a
child of this logger so verbosity can be tuned per component. Default level is
INFO; set AGENT_LOG_LEVEL=DEBUG to see full payloads.
"""
from __future__ import annotations

import json
import logging
import os
import subprocess
import sys
import time
import uuid
import glob as _glob
import math as _math
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import requests  # used to talk to Ollama, Tavily and Finnhub HTTP APIs

# Load TAVILY_API_KEY / FINNHUB_API_KEY from the project .env if python-dotenv
# is available. Search upward so it works regardless of the kernel's cwd.
try:
    from dotenv import load_dotenv, find_dotenv
    _dotenv = find_dotenv(usecwd=True)
    if _dotenv:
        load_dotenv(_dotenv)
except Exception:
    pass


# --- Logging setup -----------------------------------------------------------
AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()

class _Fmt(logging.Formatter):
    COLORS = {
        "DEBUG":    "\033[90m",   # gray
        "INFO":     "\033[36m",   # cyan
        "WARNING":  "\033[33m",   # yellow
        "ERROR":    "\033[31m",   # red
        "CRITICAL": "\033[41m",   # red bg
    }
    RESET = "\033[0m"
    def format(self, record: logging.LogRecord) -> str:
        color = self.COLORS.get(record.levelname, "")
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"{color}[{record.levelname:5s}] {short:18s} | {record.getMessage()}{self.RESET}"

_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())

log = logging.getLogger("agent")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False

# Child loggers per subsystem.
log_loop    = logging.getLogger("agent.loop")
log_ollama  = logging.getLogger("agent.ollama")
log_tool    = logging.getLogger("agent.tool")
log_research = logging.getLogger("agent.research")
log_sub     = logging.getLogger("agent.subagent")
log_todo    = logging.getLogger("agent.todo")
log_compact = logging.getLogger("agent.compact")
log_chat    = logging.getLogger("agent.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")


In [ ]:
"""
Configuration. All knobs live here.
"""

# --- Ollama HTTP endpoint ----------------------------------------------------
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:8080")

# --- Model per agent role ----------------------------------------------------
MODELS = {
    "lead":       os.environ.get("AGENT_LEAD_MODEL",       "qwen3:32b"),
    "subagent":   os.environ.get("AGENT_SUBAGENT_MODEL",   "qwen3:8b"),
    "summarizer": os.environ.get("AGENT_SUMMARIZER_MODEL", "qwen3:8b"),
}

# --- Sandbox / working dir ---------------------------------------------------
WORKDIR = Path(os.environ.get("AGENT_WORKDIR",
                              str(Path.cwd() / "sandbox"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

# --- Optional skills directory -----------------------------------------------
SKILLS_DIR = Path(os.environ.get("AGENT_SKILLS_DIR", str(WORKDIR / "skills")))

# --- State files (persisted across runs) -------------------------------------
TODO_FILE   = WORKDIR / ".agent_todo.json"
TASKS_FILE  = WORKDIR / ".agent_tasks.json"
MEMORY_FILE = WORKDIR / ".agent_memory.md"

# --- Limits ------------------------------------------------------------------
MAX_TOOL_OUTPUT     = 20_000   # truncate big tool outputs to keep ctx lean
MAX_TURNS           = 30       # hard cap on agent loop iterations per query
COMPRESS_THRESHOLD  = 40_000   # chars across all messages -> trigger compact
KEEP_RECENT         = 6        # last N messages preserved verbatim during compact
BASH_TIMEOUT_S      = 60       # subprocess timeout
REQUEST_TIMEOUT_S   = 600      # ollama HTTP timeout (32b can be slow on big ctx)

# --- Research tool config ----------------------------------------------------
TAVILY_ENDPOINT     = "https://api.tavily.com/search"
TAVILY_DEFAULT_TOPIC = os.environ.get("TAVILY_TOPIC", "finance")  # finance|news|general
TAVILY_MAX_RESULTS  = int(os.environ.get("TAVILY_MAX_RESULTS", "4"))
TAVILY_TIMEOUT_S    = 60
# When True, pull full page text and summarise each source with the summarizer
# model (slower, richer). When False, use Tavily's short content snippets.
TAVILY_SUMMARIZE    = os.environ.get("TAVILY_SUMMARIZE", "0") == "1"

FINNHUB_ENDPOINT    = "https://finnhub.io/api/v1"
FINNHUB_TIMEOUT_S   = 25

# --- Bash blocklist ----------------------------------------------------------
BASH_BLOCKLIST = [
    "rm -rf /", "sudo", "shutdown", "reboot", "mkfs",
    "> /dev/", ":(){ :|:& };:",
]

log.info(f"OLLAMA_HOST = {OLLAMA_HOST}")
log.info(f"WORKDIR     = {WORKDIR}")
log.info(f"MODELS      = {MODELS}")
log.info(f"TAVILY_API_KEY  present: {bool(os.environ.get('TAVILY_API_KEY'))}")
log.info(f"FINNHUB_API_KEY present: {bool(os.environ.get('FINNHUB_API_KEY'))}")


In [ ]:
"""
Ollama client.

Hits Ollama's native /api/chat endpoint with stream=False. Supports OpenAI-style
tool calling: pass tools=[...], the model can return message.tool_calls. Thin
layer -- no retry, no streaming; errors bubble up.
"""

def ollama_chat(
    *,
    model: str,
    messages: List[Dict[str, Any]],
    tools: Optional[List[Dict[str, Any]]] = None,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    """Call Ollama /api/chat. Returns the raw `message` dict from the response."""
    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    if tools:
        payload["tools"] = tools

    n_msgs = len(messages)
    approx_chars = sum(len(str(m.get("content", ""))) for m in messages)
    log_ollama.info(
        f"-> {model}  msgs={n_msgs}  ~{approx_chars} chars  "
        f"tools={len(tools) if tools else 0}"
    )
    log_ollama.debug(f"   request last msg: {str(messages[-1])[:300]}")

    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=REQUEST_TIMEOUT_S,
    )
    dt = time.time() - t0

    if resp.status_code != 200:
        log_ollama.error(f"<- HTTP {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

    data = resp.json()
    msg = data.get("message", {})
    tcs = msg.get("tool_calls") or []
    text_len = len(msg.get("content") or "")

    log_ollama.info(
        f"<- {model}  {dt:5.1f}s  text={text_len}ch  "
        f"tool_calls={len(tcs)}  done_reason={data.get('done_reason','?')}"
    )
    log_ollama.debug(f"   response message: {str(msg)[:500]}")
    return msg


def ollama_healthcheck() -> bool:
    """Quick sanity probe -- confirms the server is up and lists models."""
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
        tags = [m["name"] for m in r.json().get("models", [])]
        log_ollama.info(f"healthcheck OK -- {len(tags)} models available")
        for role, name in MODELS.items():
            present = any(t == name or t.startswith(name + ":") for t in tags)
            mark = "OK" if present else "MISSING"
            log_ollama.info(f"   {role:11s} {name:18s} [{mark}]")
        return True
    except Exception as e:
        log_ollama.error(f"healthcheck FAILED: {e}")
        return False

ollama_healthcheck()


In [ ]:
"""
Research tools -- Tavily (web) + Finnhub (structured market data).

Both are plain functions returning a string, so their output (including errors)
flows straight back into the conversation as a tool result. Called over raw HTTP
with `requests` -- no SDK dependencies. API keys come from the environment
(loaded from .env in the imports cell).

  tavily_search(query, max_results, topic)
      Web/news/macro research. Returns Tavily's quick answer plus per-source
      summaries WITH URLs so the model can cite. Set TAVILY_SUMMARIZE=1 to pull
      full page text and condense each source with the summarizer model.

  finnhub(action, symbol|query)
      Exact structured data. Actions:
        quote   -> live price, change, day range, prev close
        profile -> company profile (name, exchange, industry, mkt cap, shares)
        metrics -> key fundamentals (P/E, P/B, margins, ROE, 52w range, beta...)
        news    -> last ~2 weeks of company headlines (with URLs)
        search  -> resolve a company name to ticker symbol(s)
"""

# Note: `_truncate` and `MAX_TOOL_OUTPUT` are defined in the core-tools/config
# cells; they exist by the time these functions are *called*.

def _summarize_webpage(content: str) -> str:
    """Condense raw page text into an analyst-friendly summary (preserves figures)."""
    msgs = [
        {"role": "system", "content": (
            "You summarize web page content for a financial analyst. PRESERVE all "
            "numbers, dates, named entities, tickers and direct quotes that carry "
            "financial meaning. Drop boilerplate/navigation. Be concise.")},
        {"role": "user", "content": f"Summarize for financial analysis:\n\n{content[:8000]}"},
    ]
    try:
        msg = ollama_chat(model=MODELS["summarizer"], messages=msgs, tools=None)
        return (msg.get("content") or "").strip() or content[:1000]
    except Exception as e:
        log_research.warning(f"[tavily] summarise failed: {e}")
        return content[:1000]


def tool_tavily_search(query: str,
                       max_results: int = TAVILY_MAX_RESULTS,
                       topic: str = TAVILY_DEFAULT_TOPIC) -> str:
    log_research.info(f"[tavily] {query[:120]!r} (topic={topic}, n={max_results})")
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        return "Error: TAVILY_API_KEY is not set in the environment."
    if topic not in ("finance", "news", "general"):
        topic = "general"
    payload = {
        "api_key": key,
        "query": query,
        "max_results": max_results,
        "topic": topic,
        "search_depth": "advanced",
        "include_answer": True,
        "include_raw_content": TAVILY_SUMMARIZE,
    }
    try:
        r = requests.post(TAVILY_ENDPOINT, json=payload, timeout=TAVILY_TIMEOUT_S)
        if r.status_code != 200:
            return f"Error: Tavily HTTP {r.status_code}: {r.text[:300]}"
        data = r.json()
    except Exception as e:
        return f"Error: Tavily request failed: {e}"

    results = data.get("results", []) or []
    if not results and not data.get("answer"):
        return f"No results for query: {query}"

    # Deduplicate by URL.
    seen: Dict[str, dict] = {}
    for res in results:
        url = res.get("url", "")
        if url and url not in seen:
            seen[url] = res

    out = [f"Tavily search results for: {query}"]
    if data.get("answer"):
        out.append(f"\nQuick answer: {data['answer']}")
    for i, (url, res) in enumerate(seen.items(), 1):
        content = res.get("content", "") or ""
        if TAVILY_SUMMARIZE and res.get("raw_content"):
            content = _summarize_webpage(res["raw_content"])
        out.append(
            f"\n--- SOURCE {i}: {res.get('title', '(untitled)')} ---\n"
            f"URL: {url}\n{content}"
        )
    log_research.info(f"[tavily] {len(seen)} unique source(s)")
    return _truncate("\n".join(out), MAX_TOOL_OUTPUT)


# A useful subset of Finnhub's ~100 metric fields (the rest is noise for an LLM).
_FINNHUB_METRIC_KEYS = [
    "peTTM", "peAnnual", "pbAnnual", "psTTM", "epsTTM", "epsGrowthTTMYoy",
    "dividendYieldIndicatedAnnual", "marketCapitalization", "enterpriseValue",
    "52WeekHigh", "52WeekLow", "beta", "roeTTM", "roaTTM",
    "grossMarginTTM", "operatingMarginTTM", "netProfitMarginTTM",
    "currentRatioAnnual", "totalDebt/totalEquityAnnual",
    "revenueGrowthTTMYoy", "revenuePerShareTTM",
]

def tool_finnhub(action: str, symbol: str = "", query: str = "") -> str:
    log_research.info(f"[finnhub] {action} symbol={symbol!r} query={query!r}")
    key = os.environ.get("FINNHUB_API_KEY")
    if not key:
        return "Error: FINNHUB_API_KEY is not set in the environment."
    base = FINNHUB_ENDPOINT
    sym = (symbol or "").upper().strip()

    def _get(path: str, params: dict):
        params = {**params, "token": key}
        r = requests.get(f"{base}{path}", params=params, timeout=FINNHUB_TIMEOUT_S)
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
        return r.json()

    try:
        if action == "quote":
            if not sym:
                return "Error: 'quote' requires a symbol."
            d = _get("/quote", {"symbol": sym})
            if d.get("c") in (None, 0) and d.get("pc") in (None, 0):
                return f"No quote for {sym!r} (unknown symbol?). Try action='search'."
            return json.dumps({
                "symbol": sym, "current": d.get("c"), "change": d.get("d"),
                "percent_change": d.get("dp"), "high": d.get("h"),
                "low": d.get("l"), "open": d.get("o"), "prev_close": d.get("pc"),
            }, indent=2)

        if action == "profile":
            if not sym:
                return "Error: 'profile' requires a symbol."
            d = _get("/stock/profile2", {"symbol": sym})
            return json.dumps(d, indent=2) if d else f"No profile for {sym!r}."

        if action == "metrics":
            if not sym:
                return "Error: 'metrics' requires a symbol."
            d = _get("/stock/metric", {"symbol": sym, "metric": "all"})
            metric = d.get("metric", {}) or {}
            subset = {k: metric.get(k) for k in _FINNHUB_METRIC_KEYS if k in metric}
            return json.dumps({"symbol": sym, "metrics": subset or metric}, indent=2)

        if action == "news":
            if not sym:
                return "Error: 'news' requires a symbol."
            to = datetime.now().strftime("%Y-%m-%d")
            frm = (datetime.now() - timedelta(days=14)).strftime("%Y-%m-%d")
            items = _get("/company-news", {"symbol": sym, "from": frm, "to": to}) or []
            trimmed = [{
                "date": datetime.fromtimestamp(x.get("datetime", 0)).strftime("%Y-%m-%d"),
                "headline": x.get("headline"),
                "source": x.get("source"),
                "url": x.get("url"),
            } for x in items[:8]]
            return json.dumps(trimmed, indent=2) if trimmed else f"No recent news for {sym!r}."

        if action == "search":
            q = query or symbol
            if not q:
                return "Error: 'search' requires a query (company name)."
            d = _get("/search", {"q": q})
            res = (d.get("result", []) or [])[:10]
            trimmed = [{"symbol": x.get("symbol"), "description": x.get("description"),
                        "type": x.get("type")} for x in res]
            return json.dumps(trimmed, indent=2) if trimmed else f"No symbols matching {q!r}."

        return (f"Error: unknown action {action!r}. "
                "Use one of: quote, profile, metrics, news, search.")
    except Exception as e:
        return f"Error: finnhub {action} failed: {e}"

log.info("Research tools defined: tavily_search, finnhub")


In [ ]:
"""
calc -- a small, sandboxed financial-math evaluator.

The model passes a Python expression; we eval it with __builtins__ stripped and
a curated namespace: every math.* function plus finance helpers. This keeps the
agent from doing arithmetic "in its head" (where LLMs are unreliable).

Finance helpers (Excel-style sign conventions):
  npv(rate, cashflows)            net present value; cashflows[0] is t=0
  irr(cashflows)                  internal rate of return (bisection + Newton)
  fv(rate, nper, pmt, pv=0)       future value of an annuity
  pv(rate, nper, pmt, fv=0)       present value of an annuity
  pmt(rate, nper, pv, fv=0)       periodic payment for a loan/annuity

Examples:
  calc("npv(0.10, [-1000, 300, 400, 500, 600])")
  calc("irr([-1000, 300, 400, 500, 600])")
  calc("(250e9 / 100e9)")                # simple ratio
  calc("pmt(0.05/12, 360, 400000)")      # monthly payment on a 30y mortgage
"""

def _npv(rate, cashflows):
    return sum(cf / (1.0 + rate) ** i for i, cf in enumerate(cashflows))

def _irr(cashflows, guess: float = 0.1):
    lo, hi = -0.9999, 10.0
    f_lo, f_hi = _npv(lo, cashflows), _npv(hi, cashflows)
    if f_lo * f_hi <= 0:                      # bracketed -> bisection
        for _ in range(200):
            mid = (lo + hi) / 2.0
            f_mid = _npv(mid, cashflows)
            if abs(f_mid) < 1e-10:
                return mid
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo, f_lo = mid, f_mid
        return (lo + hi) / 2.0
    r = guess                                 # not bracketed -> Newton
    for _ in range(100):
        base = _npv(r, cashflows)
        deriv = (_npv(r + 1e-6, cashflows) - base) / 1e-6
        if deriv == 0:
            break
        step = base / deriv
        r -= step
        if abs(step) < 1e-10:
            return r
    return r

def _fv(rate, nper, pmt, pv=0.0):
    if rate == 0:
        return -(pv + pmt * nper)
    return -(pv * (1 + rate) ** nper + pmt * ((1 + rate) ** nper - 1) / rate)

def _pv(rate, nper, pmt, fv=0.0):
    if rate == 0:
        return -(fv + pmt * nper)
    return -(fv + pmt * ((1 + rate) ** nper - 1) / rate) / (1 + rate) ** nper

def _pmt(rate, nper, pv, fv=0.0):
    if rate == 0:
        return -(pv + fv) / nper
    return -(pv * (1 + rate) ** nper + fv) * rate / ((1 + rate) ** nper - 1)


def tool_calc(expression: str) -> str:
    log_tool.info(f"[calc] {expression[:160]}")
    ns = {k: getattr(_math, k) for k in dir(_math) if not k.startswith("_")}
    ns.update({
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
        "len": len, "pow": pow, "range": range, "sorted": sorted,
        "npv": _npv, "irr": _irr, "fv": _fv, "pv": _pv, "pmt": _pmt,
    })
    try:
        result = eval(expression, {"__builtins__": {}}, ns)
        log_tool.debug(f"[calc] = {result}")
        return str(result)
    except Exception as e:
        log_tool.error(f"[calc] {e}")
        return f"Error: {e}  (expression: {expression!r})"

log.info("calc tool defined (math.* + npv/irr/fv/pv/pmt)")


In [ ]:
"""
Core file & shell tools.

Each tool returns a string; errors become observations, not crashes. Paths are
resolved against WORKDIR and rejected if they escape it. bash() refuses the
blocklist and runs in WORKDIR. write() snapshots prior content so revert() works.
These let the agent persist analyses, scratch data, and reports to disk.
"""

SNAPSHOTS: Dict[str, Optional[str]] = {}   # {abs_path: previous_content or None}


def _safe_path(path: str) -> Path:
    p = (WORKDIR / path).resolve() if not os.path.isabs(path) else Path(path).resolve()
    try:
        p.relative_to(WORKDIR)
    except ValueError:
        raise ValueError(f"path escapes WORKDIR: {p}")
    return p


def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    if len(s) <= limit:
        return s
    return s[:limit] + f"\n... [truncated {len(s) - limit} chars]"


def tool_bash(command: str) -> str:
    log_tool.info(f"[bash] {command[:120]}")
    for bad in BASH_BLOCKLIST:
        if bad in command:
            log_tool.warning(f"[bash] BLOCKED ({bad!r}): {command[:80]}")
            return f"Error: blocked by safety policy (matched {bad!r})"
    try:
        proc = subprocess.run(
            command, shell=True, cwd=str(WORKDIR),
            capture_output=True, text=True, timeout=BASH_TIMEOUT_S,
        )
        out = (proc.stdout + proc.stderr).strip() or "(no output)"
        log_tool.info(f"[bash] exit={proc.returncode} bytes={len(out)}")
        return _truncate(out)
    except subprocess.TimeoutExpired:
        log_tool.error(f"[bash] timeout after {BASH_TIMEOUT_S}s")
        return f"Error: timeout after {BASH_TIMEOUT_S}s"
    except Exception as e:
        log_tool.error(f"[bash] exception: {e}")
        return f"Error: {e}"


def tool_read(path: str, start_line: Optional[int] = None, end_line: Optional[int] = None) -> str:
    log_tool.info(f"[read] {path} lines={start_line}:{end_line}")
    try:
        p = _safe_path(path)
        lines = p.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
        i0 = max(0, (start_line or 1) - 1)
        i1 = end_line if end_line is not None else len(lines)
        numbered = "".join(f"{i0+1+i:5d}\t{ln}" for i, ln in enumerate(lines[i0:i1]))
        return _truncate(numbered or "(empty)")
    except FileNotFoundError:
        log_tool.warning(f"[read] not found: {path}")
        return f"Error: file not found: {path}"
    except Exception as e:
        log_tool.error(f"[read] {e}")
        return f"Error: {e}"


def tool_write(path: str, content: str) -> str:
    log_tool.info(f"[write] {path} ({len(content)} chars)")
    try:
        p = _safe_path(path)
        if p.exists():
            SNAPSHOTS[str(p)] = p.read_text(encoding="utf-8", errors="replace")
            action = "updated"
        else:
            SNAPSHOTS[str(p)] = None
            action = "created"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        log_tool.info(f"[write] {action} {p} (snapshot saved)")
        return f"{action}: {p} (snapshot saved -- use revert to undo)"
    except Exception as e:
        log_tool.error(f"[write] {e}")
        return f"Error: {e}"


def tool_grep(pattern: str, path: str = ".", recursive: bool = True) -> str:
    log_tool.info(f"[grep] {pattern!r} in {path} recursive={recursive}")
    try:
        p = _safe_path(path)
        flags = ["-rn"] if recursive else ["-n"]
        proc = subprocess.run(
            ["grep", *flags, pattern, str(p)],
            capture_output=True, text=True, timeout=30,
        )
        out = (proc.stdout + proc.stderr).strip() or "(no matches)"
        return _truncate(out, 10_000)
    except subprocess.TimeoutExpired:
        return "Error: grep timeout"
    except Exception as e:
        log_tool.error(f"[grep] {e}")
        return f"Error: {e}"


def tool_glob(pattern: str) -> str:
    log_tool.info(f"[glob] {pattern}")
    full = str(WORKDIR / pattern) if not os.path.isabs(pattern) else pattern
    matches = sorted(_glob.glob(full, recursive=True))[:200]
    safe_matches = [m for m in matches if Path(m).resolve().is_relative_to(WORKDIR)]
    return "\n".join(safe_matches) if safe_matches else "(no matches)"


def tool_revert(path: str) -> str:
    log_tool.info(f"[revert] {path}")
    try:
        p = _safe_path(path)
        key = str(p)
        if key not in SNAPSHOTS:
            log_tool.warning(f"[revert] no snapshot for {p}")
            return f"Error: no snapshot for {p}"
        prev = SNAPSHOTS.pop(key)
        if prev is None:
            p.unlink(missing_ok=True)
            return f"reverted: deleted {p} (was a new file)"
        p.write_text(prev, encoding="utf-8")
        return f"reverted: {p}"
    except Exception as e:
        log_tool.error(f"[revert] {e}")
        return f"Error: {e}"

log.info("Core tools defined: bash, read, write, grep, glob, revert")


In [ ]:
"""
Tool schemas + base dispatch map.

Ollama expects OpenAI-style function schemas. DISPATCH routes a tool name to a
Python callable taking the args dict and returning a string. The BASE registry
holds research + file + shell tools; later cells extend a LEAD registry with
planning, delegation and skills (subagents keep only the BASE set on purpose).
"""

def _fn(name: str, description: str, properties: Dict[str, Any], required: List[str]) -> Dict[str, Any]:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


TOOLS_BASE: List[Dict[str, Any]] = [
    _fn(
        "tavily_search",
        ("Search the web for CURRENT information: market commentary, analyst views, "
         "SEC filings, macro/economic data, earnings coverage, company news. Your "
         "training data is stale -- use this for anything time-sensitive or that you "
         "are not certain about. Returns a quick answer plus summarized sources WITH "
         "URLs; always cite the URLs you used."),
        {
            "query":       {"type": "string", "description": "A focused search query."},
            "max_results": {"type": "integer", "description": "Default 4."},
            "topic":       {"type": "string", "enum": ["finance", "news", "general"],
                            "description": "Use 'finance' for markets/companies, 'news' for breaking events."},
        },
        ["query"],
    ),
    _fn(
        "finnhub",
        ("Fetch EXACT structured market data from Finnhub. Prefer this over web search "
         "for hard numbers. Actions: 'quote' (live price/change/day-range), 'profile' "
         "(company name, exchange, industry, market cap, shares), 'metrics' (P/E, P/B, "
         "P/S, EPS, margins, ROE/ROA, 52-week range, beta, growth), 'news' (recent "
         "headlines with URLs), 'search' (resolve a company name to its ticker)."),
        {
            "action": {"type": "string", "enum": ["quote", "profile", "metrics", "news", "search"]},
            "symbol": {"type": "string", "description": "Ticker, e.g. 'AAPL'. Required for quote/profile/metrics/news."},
            "query":  {"type": "string", "description": "Company name for action='search'."},
        },
        ["action"],
    ),
    _fn(
        "calc",
        ("Evaluate a Python math expression for financial calculations. Always use this "
         "instead of doing arithmetic yourself. Available: all math.* functions plus "
         "npv(rate, cashflows), irr(cashflows), fv(rate,nper,pmt,pv=0), "
         "pv(rate,nper,pmt,fv=0), pmt(rate,nper,pv,fv=0). "
         "Example: 'npv(0.1, [-1000, 300, 400, 500])'."),
        {"expression": {"type": "string", "description": "A single Python expression."}},
        ["expression"],
    ),
    _fn(
        "bash",
        "Run a shell command inside the sandbox WORKDIR. Stdout+stderr are returned.",
        {"command": {"type": "string", "description": "Shell command to execute."}},
        ["command"],
    ),
    _fn(
        "read",
        "Read a text file (relative to WORKDIR). Optional 1-indexed line range.",
        {
            "path":       {"type": "string"},
            "start_line": {"type": "integer", "description": "First line (1-indexed)."},
            "end_line":   {"type": "integer", "description": "Last line (inclusive)."},
        },
        ["path"],
    ),
    _fn(
        "write",
        "Write content to a file (e.g. save an analysis or report). Snapshotted; revert to undo.",
        {"path": {"type": "string"}, "content": {"type": "string"}},
        ["path", "content"],
    ),
    _fn(
        "grep",
        "Regex search across files under a path.",
        {
            "pattern":   {"type": "string"},
            "path":      {"type": "string", "description": "Defaults to '.'"},
            "recursive": {"type": "boolean"},
        },
        ["pattern"],
    ),
    _fn(
        "glob",
        "Find files matching a glob pattern, e.g. '**/*.csv'.",
        {"pattern": {"type": "string"}},
        ["pattern"],
    ),
    _fn(
        "revert",
        "Restore a file to its state before the most recent write.",
        {"path": {"type": "string"}},
        ["path"],
    ),
]

DISPATCH_BASE: Dict[str, Callable[[Dict[str, Any]], str]] = {
    "tavily_search": lambda a: tool_tavily_search(
        a["query"], a.get("max_results", TAVILY_MAX_RESULTS),
        a.get("topic", TAVILY_DEFAULT_TOPIC)),
    "finnhub": lambda a: tool_finnhub(a["action"], a.get("symbol", ""), a.get("query", "")),
    "calc":   lambda a: tool_calc(a["expression"]),
    "bash":   lambda a: tool_bash(a["command"]),
    "read":   lambda a: tool_read(a["path"], a.get("start_line"), a.get("end_line")),
    "write":  lambda a: tool_write(a["path"], a["content"]),
    "grep":   lambda a: tool_grep(a["pattern"], a.get("path", "."), a.get("recursive", True)),
    "glob":   lambda a: tool_glob(a["pattern"]),
    "revert": lambda a: tool_revert(a["path"]),
}

log.info(f"Base tool registry: {list(DISPATCH_BASE)}")


In [ ]:
"""
Agent loop -- perception -> action -> observation -> repeat.

  1. PERCEPTION  : send the conversation so far to the model
  2. ACTION      : if the model emitted tool_calls, run them
  3. OBSERVATION : feed the tool results back as new messages

Terminates the moment the model returns a message with no tool_calls -- that
final text is the answer. MAX_TURNS caps runaway loops. `messages` is mutated
in place so the caller can inspect the full transcript.
"""

def _run_one_tool_call(tc: Dict[str, Any], dispatch: Dict[str, Callable]) -> Dict[str, Any]:
    fn   = tc.get("function", {}) or {}
    name = fn.get("name", "")
    args = fn.get("arguments", {}) or {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}

    arg_preview = ", ".join(f"{k}={str(v)[:40]!r}" for k, v in args.items())
    log_tool.info(f"-> {name}({arg_preview})")

    if name not in dispatch:
        result = f"[error] Unknown tool: {name!r}. Available: {sorted(dispatch)}"
        log_tool.warning(result)
    else:
        try:
            result = dispatch[name](args)
        except Exception as e:
            result = f"[error] Tool {name!r} raised {type(e).__name__}: {e}"
            log_tool.exception(result)

    result = _truncate(str(result), MAX_TOOL_OUTPUT)
    log_tool.debug(f"<- {name} result ({len(result)} chars):\n{result}")
    return {"role": "tool", "name": name, "content": result}


def agent_loop(
    messages: List[Dict[str, Any]],
    tools:    List[Dict[str, Any]],
    dispatch: Dict[str, Callable],
    model:    str = MODELS["lead"],
    max_turns: int = MAX_TURNS,
    label:    str = "lead",
) -> str:
    tool_names = [t["function"]["name"] for t in tools]
    log_loop.info(f"[{label}] starting (model={model}, max_turns={max_turns}, tools={tool_names})")

    final_text = ""
    for turn in range(1, max_turns + 1):
        log_loop.info(f"[{label}] turn {turn}/{max_turns}  msgs={len(messages)}")

        # ---- PERCEPTION ----
        msg = ollama_chat(model=model, messages=messages, tools=tools)
        messages.append(msg)

        tool_calls = msg.get("tool_calls") or []
        text       = (msg.get("content") or "").strip()

        # ---- TERMINATION ----
        if not tool_calls:
            final_text = text
            preview = final_text[:120] + ("..." if len(final_text) > 120 else "")
            log_loop.info(f"[{label}] DONE on turn {turn} -- no tool_calls. final={preview!r}")
            return final_text

        # ---- ACTION + OBSERVATION ----
        log_loop.info(f"[{label}] turn {turn}: model requested {len(tool_calls)} tool call(s)")
        if text:
            log_loop.debug(f"[{label}] assistant said (alongside tool_calls): {text[:200]}")

        for tc in tool_calls:
            messages.append(_run_one_tool_call(tc, dispatch))

    log_loop.warning(f"[{label}] hit MAX_TURNS={max_turns} without termination.")
    return final_text or "[agent_loop] max turns reached without a terminal response"

log.info("Agent loop ready: agent_loop(messages, tools, dispatch, model=..., label=...)")


In [ ]:
"""
Subagents -- isolated context for messy subtasks.

When the lead hits a noisy subtask (research a single company across several
sources, dig through a filing) it calls spawn_subagent(prompt=...). That runs a
fresh agent_loop with an empty history, the smaller subagent model, and the BASE
tool set -- which here INCLUDES tavily_search, finnhub and calc, so a subagent
can do real research. It does NOT include spawn_subagent itself (no recursion).
Only the subagent's final text returns to the lead; its intermediate turns stay
out of the lead's context.
"""

SUBAGENT_SYSTEM = (
    f"You are a focused financial-research subagent working in a sandbox at {WORKDIR}. "
    "You have tools: tavily_search (web/news), finnhub (live quotes/fundamentals/news), "
    "calc (financial math), and file/shell tools. You are given a single, self-contained "
    "research subtask. Gather the needed data, run any required calculations, and when "
    "confident reply with a concise, FACT-DENSE summary that includes concrete numbers "
    "and the source URLs you relied on -- that final message is what gets returned to the "
    "lead agent. Do not ask clarifying questions; make a reasonable assumption and proceed. "
    "Never invent figures; if data is unavailable, say so explicitly."
)


def run_subagent(prompt: str) -> str:
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- prompt: {prompt[:120]!r}")

    sub_messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user",   "content": prompt},
    ]
    try:
        result = agent_loop(
            messages=sub_messages,
            tools=TOOLS_BASE,
            dispatch=DISPATCH_BASE,
            model=MODELS["subagent"],
            label=f"sub:{sub_id}",
        )
    except Exception as e:
        log_sub.exception(f"[sub:{sub_id}] crashed: {e}")
        return f"[subagent error] {type(e).__name__}: {e}"

    log_sub.info(f"[sub:{sub_id}] done -- {len(result)} chars: "
                 f"{result[:120]!r}{'...' if len(result) > 120 else ''}")
    return result


# --- Register spawn_subagent on a LEAD registry (extends, does not mutate BASE) ---
TOOLS_LEAD: List[Dict[str, Any]] = TOOLS_BASE + [
    _fn(
        "spawn_subagent",
        ("Delegate a self-contained research subtask to a fresh subagent (which has the "
         "same research tools). Use for deep single-company digs or noisy multi-source "
         "gathering that would clutter your own context. Returns only the subagent's "
         "final summary."),
        {"prompt": {"type": "string",
                    "description": "Detailed self-contained instructions for the subagent."}},
        ["prompt"],
    ),
]

DISPATCH_LEAD: Dict[str, Callable[[Dict[str, Any]], str]] = {
    **DISPATCH_BASE,
    "spawn_subagent": lambda a: run_subagent(a["prompt"]),
}

log.info(f"Subagent ready. Lead tool registry now: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Todo planning -- the agent's working plan for the CURRENT request.

A lightweight, ordered checklist written at the start of a complex request and
checked off as steps complete. Forces an explicit plan and gives visibility into
intent. State lives in TODO_FILE so it persists across restarts.

  todo_write(items)          -- replace the whole list (start as 'pending')
  todo_read()                -- show the current list with statuses
  todo_update(index, status) -- mark one item pending/in_progress/done
"""

_TODO_STATUSES = ("pending", "in_progress", "done")


def _load_todos() -> List[Dict[str, Any]]:
    if not TODO_FILE.exists():
        return []
    try:
        return json.loads(TODO_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        log_todo.warning(f"todo file unreadable, starting empty: {e}")
        return []


def _save_todos(items: List[Dict[str, Any]]) -> None:
    TODO_FILE.write_text(json.dumps(items, indent=2), encoding="utf-8")


def run_todo_write(items: List[str]) -> str:
    log_todo.info(f"[todo_write] {len(items)} items")
    todos = [{"index": i, "text": t, "status": "pending"} for i, t in enumerate(items)]
    _save_todos(todos)
    return f"Wrote {len(todos)} todos:\n" + run_todo_read()


def run_todo_read() -> str:
    todos = _load_todos()
    if not todos:
        return "(no todos)"
    marks = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}
    lines = [f"{marks.get(t['status'], '[?]')} {t['index']}: {t['text']}" for t in todos]
    return "\n".join(lines)


def run_todo_update(index: int, status: str) -> str:
    log_todo.info(f"[todo_update] #{index} -> {status}")
    if status not in _TODO_STATUSES:
        return f"Error: status must be one of {_TODO_STATUSES}, got {status!r}"
    todos = _load_todos()
    for t in todos:
        if t["index"] == index:
            t["status"] = status
            _save_todos(todos)
            return f"Todo #{index} -> {status}\n" + run_todo_read()
    return f"Error: no todo with index {index}"


TOOLS_LEAD += [
    _fn(
        "todo_write",
        ("Set the agent's working todo list. Call this at the start of any multi-step "
         "financial question to lay out your plan. Replaces any existing list."),
        {"items": {"type": "array", "items": {"type": "string"},
                   "description": "Ordered list of plain-text steps."}},
        ["items"],
    ),
    _fn("todo_read", "Show the current todo list with statuses.", {}, []),
    _fn(
        "todo_update",
        "Update the status of one todo. Use as you make progress.",
        {
            "index":  {"type": "integer", "description": "0-based index from todo_write order."},
            "status": {"type": "string",  "enum": list(_TODO_STATUSES)},
        },
        ["index", "status"],
    ),
]

DISPATCH_LEAD.update({
    "todo_write":  lambda a: run_todo_write(a["items"]),
    "todo_read":   lambda a: run_todo_read(),
    "todo_update": lambda a: run_todo_update(a["index"], a["status"]),
})

log.info(f"Todo tools registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Context compaction -- making room when the conversation gets long.

  Layer 1 (verbatim)  : keep the last KEEP_RECENT messages exactly.
  Layer 2 (summarise) : condense everything older with the summariser model.
  Layer 3 (persist)   : write that summary to MEMORY_FILE so it survives restarts.

Trigger: total character count across messages crosses COMPRESS_THRESHOLD. The
summariser is told to preserve decisions, figures, tickers, URLs and pending work.
"""

def _estimate_size(messages: List[Dict[str, Any]]) -> int:
    return sum(len(str(m.get("content", "") or "")) for m in messages)


def _render_for_summary(messages: List[Dict[str, Any]]) -> str:
    parts = []
    for m in messages:
        role = m.get("role", "?")
        content = str(m.get("content", "") or "")
        if m.get("tool_calls"):
            names = [tc.get("function", {}).get("name", "?") for tc in m["tool_calls"]]
            content = (content + f"  (called tools: {', '.join(names)})").strip()
        parts.append(f"[{role}] {content}")
    return "\n\n".join(parts)


def _summarize(messages: List[Dict[str, Any]]) -> str:
    transcript = _render_for_summary(messages)
    log_compact.info(f"summarising {len(messages)} msgs (~{len(transcript)} chars) "
                     f"with {MODELS['summarizer']}")
    summary_messages = [
        {"role": "system", "content": (
            "You are a context compressor for a financial-analysis agent. Condense the "
            "conversation below into a concise summary. PRESERVE: figures and their units, "
            "tickers, source URLs, calculations performed, decisions made, and any pending "
            "or unfinished work. DROP pleasantries and redundancy. Write prose, not a list.")},
        {"role": "user", "content": f"Summarise this history:\n\n{transcript[:20000]}"},
    ]
    msg = ollama_chat(model=MODELS["summarizer"], messages=summary_messages, tools=None)
    summary = (msg.get("content") or "").strip()
    log_compact.info(f"summary produced ({len(summary)} chars)")
    return summary


def maybe_compress(messages: List[Dict[str, Any]]) -> bool:
    size = _estimate_size(messages)
    if size < COMPRESS_THRESHOLD:
        return False
    if len(messages) <= KEEP_RECENT + 1:
        return False

    log_compact.info(f"context ~{size} chars >= {COMPRESS_THRESHOLD} -- compressing")

    head = []
    body = messages
    if messages and messages[0].get("role") == "system":
        head = [messages[0]]
        body = messages[1:]

    old    = body[:-KEEP_RECENT]
    recent = body[-KEEP_RECENT:]
    if not old:
        return False

    summary = _summarize(old)

    try:
        MEMORY_FILE.write_text(f"# Agent context memory\n\n{summary}\n", encoding="utf-8")
        log_compact.info(f"summary persisted -> {MEMORY_FILE}")
    except OSError as e:
        log_compact.error(f"could not persist memory: {e}")

    messages.clear()
    messages.extend(head)
    messages.append({"role": "user",
                     "content": f"[Summary of earlier turns]:\n\n{summary}"})
    messages.append({"role": "assistant",
                     "content": "Understood. I've integrated the summary of our earlier progress."})
    messages.extend(recent)

    log_compact.info(f"compressed {len(old)} msgs into 1 summary; "
                     f"history now {len(messages)} msgs")
    return True

log.info("Context compaction ready: maybe_compress(messages)")


In [ ]:
"""
Task graph -- a persistent, dependency-aware backlog.

Heavier sibling of the todo list: durable units of work with stable IDs,
dependencies, priority and a result field, persisted to TASKS_FILE. task_next
computes the next unblocked task (a pending task whose deps are all done), so
"what next?" is a graph query rather than a judgement call. Good for
multi-session financial workstreams (e.g. a coverage list of names to analyse).

  {"id","description","status","priority","depends_on":[...],"result"}
  statuses: pending | in_progress | done | failed
"""

_TASK_STATUSES = ("pending", "in_progress", "done", "failed")


def _load_tasks() -> List[Dict[str, Any]]:
    if not TASKS_FILE.exists():
        return []
    try:
        return json.loads(TASKS_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        log.warning(f"[task] tasks file unreadable, starting empty: {e}")
        return []


def _save_tasks(tasks: List[Dict[str, Any]]) -> None:
    TASKS_FILE.write_text(json.dumps(tasks, indent=2), encoding="utf-8")


def run_task_create(description: str, depends_on: Optional[List[str]] = None,
                    priority: str = "medium") -> str:
    tasks = _load_tasks()
    task_id = uuid.uuid4().hex[:8]
    tasks.append({
        "id": task_id, "description": description, "status": "pending",
        "priority": priority, "depends_on": depends_on or [], "result": "",
    })
    _save_tasks(tasks)
    log.info(f"[task] created {task_id}: {description[:60]} (deps={depends_on or []})")
    return f"Created task {task_id}: {description}"


def run_task_list() -> str:
    tasks = _load_tasks()
    if not tasks:
        return "(no tasks)"
    lines = []
    for t in tasks:
        deps = f" needs={','.join(t['depends_on'])}" if t.get("depends_on") else ""
        lines.append(f"[{t['id']}] {t['status']:11s} {t['priority']:6s}{deps}  {t['description']}")
    return "\n".join(lines)


def run_task_update(task_id: str, status: str, result: str = "") -> str:
    if status not in _TASK_STATUSES:
        return f"Error: status must be one of {_TASK_STATUSES}, got {status!r}"
    tasks = _load_tasks()
    for t in tasks:
        if t["id"].startswith(task_id):
            t["status"] = status
            if result:
                t["result"] = result
            _save_tasks(tasks)
            log.info(f"[task] {t['id']} -> {status}")
            return f"Task {t['id']} -> {status}"
    return f"Error: no task matching id {task_id!r}"


def run_task_next() -> str:
    tasks = _load_tasks()
    done_ids = {t["id"] for t in tasks if t["status"] == "done"}
    for t in tasks:
        if t["status"] != "pending":
            continue
        if all(dep in done_ids for dep in t.get("depends_on", [])):
            log.info(f"[task] next -> {t['id']}")
            return f"Next: [{t['id']}] ({t['priority']}) {t['description']}"
    return "No unblocked tasks. Everything is done, in progress, or waiting on a dependency."


TOOLS_LEAD += [
    _fn(
        "task_create",
        ("Create a persistent task in the dependency graph. Use for multi-session or "
         "dependency-ordered work (heavier than todo_write)."),
        {
            "description": {"type": "string"},
            "depends_on":  {"type": "array", "items": {"type": "string"},
                            "description": "IDs of tasks that must be done first."},
            "priority":    {"type": "string", "enum": ["high", "medium", "low"]},
        },
        ["description"],
    ),
    _fn("task_list", "List all tasks with IDs, status, priority and dependencies.", {}, []),
    _fn(
        "task_update",
        "Update a task's status (and optionally record its result).",
        {
            "task_id": {"type": "string", "description": "Full ID or unique prefix."},
            "status":  {"type": "string", "enum": list(_TASK_STATUSES)},
            "result":  {"type": "string", "description": "Optional summary of work done."},
        },
        ["task_id", "status"],
    ),
    _fn("task_next", "Ask the graph for the next unblocked task (all dependencies satisfied).", {}, []),
]

DISPATCH_LEAD.update({
    "task_create": lambda a: run_task_create(a["description"], a.get("depends_on"),
                                             a.get("priority", "medium")),
    "task_list":   lambda a: run_task_list(),
    "task_update": lambda a: run_task_update(a["task_id"], a["status"], a.get("result", "")),
    "task_next":   lambda a: run_task_next(),
})

log.info(f"Task graph registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Skill loading -- lazy, on-demand domain knowledge.

A "skill" is a folder under SKILLS_DIR containing a SKILL.md (a how-to: an
analysis checklist, a valuation method, gotchas). The system prompt gets only a
cheap one-line INDEX of skill names; the model calls load_skill(name) to pull a
full guide into context when relevant. SKILLS_DIR is optional -- everything
degrades to "(no skills available)".

  list_skills()    -- cheap index (name + one-line summary)
  load_skill(name) -- full SKILL.md body for one skill
"""

def _skill_dir(name: str) -> Optional[Path]:
    if not name or "/" in name or ".." in name:
        return None
    d = (SKILLS_DIR / name).resolve()
    try:
        d.relative_to(SKILLS_DIR.resolve())
    except ValueError:
        return None
    return d if (d / "SKILL.md").is_file() else None


def run_list_skills() -> str:
    if not SKILLS_DIR.is_dir():
        return "(no skills available)"
    entries = []
    for child in sorted(SKILLS_DIR.iterdir()):
        md = child / "SKILL.md"
        if not md.is_file():
            continue
        summary = ""
        for line in md.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.strip():
                summary = line.lstrip("# ").strip()
                break
        entries.append(f"- {child.name}: {summary}")
    if not entries:
        return "(no skills available)"
    return "\n".join(entries)


def run_load_skill(name: str) -> str:
    log.info(f"[skill] load {name!r}")
    d = _skill_dir(name)
    if d is None:
        return f"Error: no skill named {name!r}. Available:\n{run_list_skills()}"
    body = (d / "SKILL.md").read_text(encoding="utf-8", errors="replace")
    return _truncate(body)


def skills_index_for_prompt() -> str:
    idx = run_list_skills()
    if idx == "(no skills available)":
        return ""
    return ("\n\nAvailable skills (call load_skill(name) to read the full guide "
            "when relevant):\n" + idx)


TOOLS_LEAD += [
    _fn("list_skills", "List available skill guides (name + one-line summary each).", {}, []),
    _fn(
        "load_skill",
        ("Load the full text of one skill guide by name. Call when a skill from "
         "list_skills looks relevant to the current task."),
        {"name": {"type": "string", "description": "Skill folder name from list_skills."}},
        ["name"],
    ),
]

DISPATCH_LEAD.update({
    "list_skills": lambda a: run_list_skills(),
    "load_skill":  lambda a: run_load_skill(a["name"]),
})

log.info(f"Skill loading registered. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
chat() -- the user-facing entry point.

Wires everything into one function. Per call:
  1. (first call) build the lead SYSTEM prompt (incl. live skills index), seed HISTORY.
  2. append the user's query.
  3. run agent_loop with the FULL lead toolset (TOOLS_LEAD / DISPATCH_LEAD).
  4. maybe_compress(HISTORY) so long sessions stay within budget.
  5. return the final answer (also printed).

HISTORY persists across calls -- a sequence of chat() calls is one continuous
conversation. reset_chat() wipes it and the on-disk memory.
"""

LEAD_SYSTEM = (
    f"You are a rigorous financial research and analysis assistant operating in a sandbox "
    f"at {WORKDIR}.\n\n"
    "TOOLS:\n"
    "- finnhub: EXACT live data -- quotes, fundamentals (P/E, margins, ROE, market cap), "
    "company profile, recent news, ticker lookup. Prefer this for hard numbers.\n"
    "- tavily_search: web/news/macro research -- filings, analyst views, economic data, "
    "qualitative context. Returns sources with URLs.\n"
    "- calc: ALL arithmetic and finance math (npv, irr, fv, pv, pmt). Never compute by hand.\n"
    "- files (read/write/grep/glob/revert) and bash: persist analyses, reports, scratch data.\n"
    "- planning: todo_write/read/update (this request) and task_create/list/update/next "
    "(durable multi-step workstreams).\n"
    "- spawn_subagent: delegate a deep single-name dig or noisy multi-source gathering.\n"
    "- list_skills/load_skill: on-demand analysis playbooks.\n\n"
    "OPERATING PRINCIPLES:\n"
    "1. For any multi-step question, call todo_write FIRST to lay out your plan, then work "
    "through it, updating statuses as you go.\n"
    "2. Your training data is STALE. For any current price, fundamental, rate, or recent "
    "event, GET THE DATA via finnhub or tavily_search -- do not answer from memory.\n"
    "3. To resolve a company name to a ticker, use finnhub action='search'.\n"
    "4. Do ALL arithmetic with calc. Show the expression you evaluated.\n"
    "5. NEVER fabricate a figure. If data is unavailable, say so and explain the limitation.\n"
    "6. Always CITE sources: include the URLs (tavily) or note 'Finnhub' and the as-of "
    "date/time for live data.\n"
    "7. Delegate noisy per-company research to a subagent so your own context stays clean.\n"
    "8. When fully answered, reply with a clear, well-structured final message: the bottom "
    "line first, then the supporting numbers, assumptions, and sources. STOP calling tools.\n"
    "9. End with a one-line disclaimer that this is informational, not financial advice.\n"
)

HISTORY: List[Dict[str, Any]] = []


def _ensure_history() -> None:
    if not HISTORY:
        system = LEAD_SYSTEM + skills_index_for_prompt()
        HISTORY.append({"role": "system", "content": system})
        log_chat.info(f"history seeded (system prompt {len(system)} chars)")


def reset_chat() -> None:
    HISTORY.clear()
    try:
        MEMORY_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_chat.info("chat history reset")


def chat(query: str) -> str:
    """Drive the agent for one user query. Returns (and prints) the final answer."""
    _ensure_history()
    log_chat.info(f"=== USER: {query[:200]!r} ===")
    HISTORY.append({"role": "user", "content": query})

    answer = agent_loop(
        messages=HISTORY,
        tools=TOOLS_LEAD,
        dispatch=DISPATCH_LEAD,
        model=MODELS["lead"],
        label="lead",
    )

    if maybe_compress(HISTORY):
        log_chat.info("history compacted after this turn")

    log_chat.info(f"=== ASSISTANT ({len(answer)} chars) ===")
    print("\n" + "=" * 70 + "\nASSISTANT:\n" + "=" * 70)
    print(answer)
    return answer

log.info("chat() ready. Call chat('...') to drive the agent; reset_chat() to start over.")


In [ ]:
"""
Smoke test -- drive the whole pipeline end to end against live services.

Exercises: healthcheck -> chat() -> agent_loop -> ollama_chat -> tool dispatch
(finnhub + calc, and likely todo_*). We ask a small, multi-step, verifiable
financial question. The assertions are soft: live data changes, and the run
depends on network + valid API keys, so we check that the agent USED its tools
rather than hard-coding an expected number.

Set TAVILY_SUMMARIZE=1 (env) before importing if you want full-page summaries.
"""
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix the config cell first."

print("\nAPI keys -> "
      f"TAVILY: {'set' if os.environ.get('TAVILY_API_KEY') else 'MISSING'}, "
      f"FINNHUB: {'set' if os.environ.get('FINNHUB_API_KEY') else 'MISSING'}")

# Quick direct connectivity probes (independent of the LLM) so failures are obvious.
print("\n-- direct tool probes --")
print("finnhub quote AAPL:", tool_finnhub("quote", "AAPL")[:200])
print("calc npv:", tool_calc("round(npv(0.10, [-1000, 300, 400, 500, 600]), 2)"))

reset_chat()
answer = chat(
    "Compare Apple (AAPL) and Microsoft (MSFT): pull each company's current price and "
    "trailing P/E from finnhub, then use calc to compute the ratio of their P/E values "
    "and the percentage difference. State each figure with its source and as-of context, "
    "and tell me which looks more expensive on a P/E basis."
)

print("\n" + "-" * 70)
print(f"history now holds {len(HISTORY)} messages (~{_estimate_size(HISTORY)} chars).")
print(f"todo file exists: {TODO_FILE.exists()}")
